# 02_optimization_and_scheduling
Smart scheduler that shifts flexible workloads to time steps with low outdoor temperature and humidity, optimizing for minimum energy and water cost under SLA constraints.

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

base_dir = r"C:\Users\RANADEEP\Documents\IBM Project\Alibaba_v2018"
processed_dir = os.path.join(base_dir, "02_processed_data")


In [2]:
# Load enhanced dataset
df = pd.read_csv(os.path.join(processed_dir, 'workload_timeseries_enhanced.csv'))
df['datetime'] = pd.to_datetime(df['datetime'])

# Re-train models quickly
features = df[['cpu_util_percent', 'temperature_2m', 'humidity_2m']]
y_power = df['total_power_kw']
y_water = df['dynamic_water_liters']

model_power = RandomForestRegressor(n_estimators=50, random_state=42)
model_power.fit(features, y_power)

model_water = RandomForestRegressor(n_estimators=50, random_state=42)
model_water.fit(features, y_water)
print('Models trained.')


Models trained.


## Optimization Scenario
We extract a 24-hour window. A flexible batch job requires +20% CPU for 3 hours. SLA constraint: It must finish within this 24-hour window.
Cost metric = Energy (kWh) + Water (Liters).
Baseline: Run job immediately (hours 0, 1, 2).
Optimized: Find the start hour that minimizes total cost.

In [3]:
# Extract first 24 hours
df_24h = df.head(24).copy().reset_index(drop=True)
JOB_DURATION = 3
JOB_CPU_LOAD = 20.0  # +20% CPU
SLA_WINDOW = 24

# Weights for cost function
COST_PER_KWH = 0.10
COST_PER_LITER = 0.005

def evaluate_schedule(start_hour, df_window):
    temp_df = df_window.copy()
    # Add job CPU to the scheduled hours
    for i in range(start_hour, start_hour + JOB_DURATION):
        temp_df.loc[i, 'cpu_util_percent'] = min(100.0, temp_df.loc[i, 'cpu_util_percent'] + JOB_CPU_LOAD)
    
    # Predict Power and Water using the updated features
    sched_features = temp_df[['cpu_util_percent', 'temperature_2m', 'humidity_2m']]
    pred_pwr = model_power.predict(sched_features)
    pred_wtr = model_water.predict(sched_features)
    
    total_energy = pred_pwr.sum() # Assuming 1-hour time steps, kW * 1h = kWh
    total_water = pred_wtr.sum()
    
    total_cost = (total_energy * COST_PER_KWH) + (total_water * COST_PER_LITER)
    return total_energy, total_water, total_cost

# Baseline: start at hour 0
base_energy, base_water, base_cost = evaluate_schedule(0, df_24h)

# Evaluate all valid start times within SLA
best_start = 0
best_cost = float('inf')
best_energy = 0
best_water = 0

results = []
for h in range(SLA_WINDOW - JOB_DURATION + 1):
    eng, wtr, cost = evaluate_schedule(h, df_24h)
    results.append({'start_hour': h, 'energy': eng, 'water': wtr, 'cost': cost})
    if cost < best_cost:
        best_cost = cost
        best_start = h
        best_energy = eng
        best_water = wtr


In [4]:
# Documentation and Savings
print("--- Optimization Results ---")
print(f"Baseline (Start Hour 0): Energy={base_energy:.2f} kWh, Water={base_water:.2f} L, Cost=${base_cost:.4f}")
print(f"Optimized (Start Hour {best_start}): Energy={best_energy:.2f} kWh, Water={best_water:.2f} L, Cost=${best_cost:.4f}")

energy_saved = base_energy - best_energy
water_saved = base_water - best_water
cost_saved = base_cost - best_cost

print(f"\nTotal Savings:")
print(f"- Energy Saved: {energy_saved:.2f} kWh ({(energy_saved/base_energy)*100:.2f}%)")
print(f"- Water Saved: {water_saved:.2f} Liters ({(water_saved/base_water)*100:.2f}%)")
print(f"- Cost Reduced: ${cost_saved:.4f}")

print("\nSLA & Temperature Constraints:")
print(f"- The job finishes at hour {best_start + JOB_DURATION}, which is well within the {SLA_WINDOW}-hour SLA deadline.")
print(f"- Workload was shifted to hour {best_start} because average temperature was {df_24h.loc[best_start:best_start+JOB_DURATION-1, 'temperature_2m'].mean():.2f}°C compared to {df_24h.loc[0:JOB_DURATION-1, 'temperature_2m'].mean():.2f}°C at baseline.")


--- Optimization Results ---
Baseline (Start Hour 0): Energy=4.94 kWh, Water=6.28 L, Cost=$0.5255
Optimized (Start Hour 5): Energy=4.94 kWh, Water=6.27 L, Cost=$0.5255

Total Savings:
- Energy Saved: -0.00 kWh (-0.00%)
- Water Saved: 0.00 Liters (0.01%)
- Cost Reduced: $0.0000

SLA & Temperature Constraints:
- The job finishes at hour 8, which is well within the 24-hour SLA deadline.
- Workload was shifted to hour 5 because average temperature was 14.50°C compared to 14.50°C at baseline.
